# Bone marrow integration with regularizedvi

This tutorial demonstrates how to use **regularizedvi** (`AmbientRegularizedSCVI`) for single-cell RNA-seq integration, using the NeurIPS 2021 bone marrow multiome dataset (69k cells, 13 batches, 4 sites, 10 donors).

The key idea is to adapt [cell2location](https://doi.org/10.1038/s41587-021-01139-4)/[cell2fate](https://doi.org/10.1038/s41592-025-02608-3) modelling principles to scVI:

1. **Ambient RNA correction** — per-gene, per-batch additive background captures ambient RNA contamination
2. **Dispersion regularisation** — Exponential containment prior prevents NB collapse
3. **Batch-free decoder** — batch correction through additive background and categorical covariates
4. **Learned library size** — with constrained prior (observed totals include ambient RNA)

## Data download

The data is available from the Open Problems benchmark:

```bash
aws s3 sync s3://openproblems-bio/public/post_competition/multiome/ ./data/ --no-sign-request
```

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
import scipy
import torch

import regularizedvi

# Use high-precision matmul for better numerical stability on GPU
torch.set_float32_matmul_precision("high")

rcParams["pdf.fonttype"] = 42  # enables correct plotting of text for publication figures

## 1. Load data

Load the bone marrow multiome dataset and extract the GEX (gene expression) modality.

In [ ]:
sc_data_folder = "/nfs/team283/vk7/sanger_projects/large_data/bone_marrow/"
results_folder = "/nfs/team283/vk7/sanger_projects/cell2state_embryo/results/bone_marrow/"

adata = sc.read(f"{sc_data_folder}bmmc_multiome_multivi_neurips21_curated.h5ad")

# Extract gene expression only
adata = adata[:, adata.var["feature_types"] == "GEX"].copy()
adata.var["SYMBOL"] = adata.var_names.values.copy()
adata.var_names = adata.var["gene_ids"].values.astype(str).copy()
adata.X = adata.layers["counts"]

print(adata)
print(f"\nBatches: {adata.obs['batch'].nunique()}")
print(f"Sites: {adata.obs['site'].nunique()}")
print(f"Donors: {adata.obs['donor'].nunique()}")

## 2. Quality control

In [ ]:
# Batch and cell type composition
for c in ["batch", "site", "donor", "l2_cell_type"]:
    print(adata.obs[c].value_counts())
    print()

In [ ]:
# QC distributions
rcParams["figure.figsize"] = 5, 2
plt.hist(np.log10(adata.obs["GEX_n_counts"]), bins=100)
plt.xlabel("log10(UMI counts)")
plt.title("Total UMI counts per cell")
plt.show()

plt.hist(np.log10(adata.obs["GEX_n_genes"]), bins=100)
plt.xlabel("log10(genes detected)")
plt.title("Genes detected per cell")
plt.show()

In [ ]:
# Mitochondrial fraction
adata.var["mt"] = [gene.startswith("MT-") for gene in adata.var["SYMBOL"]]
adata.obs["mt_frac"] = adata[:, adata.var["mt"].tolist()].X.sum(1).A.squeeze() / adata.obs["GEX_n_counts"]

plt.hist(adata.obs["mt_frac"], bins=50)
plt.xlabel("Mitochondrial fraction")
plt.title("MT fraction per cell")
plt.show()

### Gene selection for QC

Select informative genes using `regularizedvi.utils.filter_genes` (adapted from cell2location). These genes are used for doublet detection below; the actual subsetting of `adata` happens after cell filtering.

In [ ]:
from regularizedvi.utils import filter_genes

selected = filter_genes(
    adata,
    cell_count_cutoff=15,
    cell_percentage_cutoff2=0.01,
    nonz_mean_cutoff=1.03,
)

print(f"Selected {selected.sum()} genes for QC and doublet detection")

In [ ]:
import scrublet as scr
import gc

sample_col = "batch"
adata.obs["scrublet_score"] = 0.0

for s in adata.obs[sample_col].unique():
    print(s)
    ind = adata.obs[sample_col] == s
    adata_sample = adata[ind, selected].copy()
    if adata_sample.n_obs > 2:
        scrub = scr.Scrublet(adata_sample.X, n_neighbors=15)
        doublet_scores, predicted_doublets = scrub.scrub_doublets(
            verbose=False, n_prin_comps=40, min_gene_variability_pctl=30
        )
        adata.obs.loc[ind, "scrublet_score"] = doublet_scores
        del adata_sample, doublet_scores, predicted_doublets, scrub
        gc.collect()

plt.hist(adata.obs["scrublet_score"], bins=50)
(adata.obs["scrublet_score"] > 0.18).mean()

In [ ]:
# Cell filtering
n_before = adata.n_obs
adata = adata[
    (adata.obs["GEX_n_genes"] > 500)
    & (adata.obs["GEX_n_counts"] > 1000)
    & (adata.obs["GEX_n_counts"] < 80000)
    & (adata.obs["GEX_n_genes"] < 10000)
    & (adata.obs["ATAC_atac_fragments"] > 1000)
    & (adata.obs["ATAC_atac_fragments"] < 100000)
    & (adata.obs["mt_frac"] < 0.20)
    & (adata.obs["scrublet_score"] < 0.18),
    :,
]
print(f"Filtered {n_before} → {adata.n_obs} cells ({n_before - adata.n_obs} removed)")

## 3. Gene filtering

Recompute gene selection on the filtered cells — cell composition affects detection rates, so we re-run `filter_genes` after QC filtering.

In [ ]:
selected = filter_genes(
    adata,
    cell_count_cutoff=15,
    cell_percentage_cutoff2=0.01,
    nonz_mean_cutoff=1.03,
)
adata = adata[:, selected].copy()

print(f"Selected {adata.n_vars} genes")

## 4. Model setup and training

### Setup anndata

- `batch_key='batch'` — identifies biological samples. Each sample gets its own ambient RNA background ($b_{g,s}$) and overdispersion ($\theta_{g,s}$).
- `categorical_covariate_keys=['site', 'donor']` — known technical/biological covariates modelled as multiplicative effects (like $p_{tg}$ in cell2location). These handle systematic differences between processing sites and donors.

In [ ]:
adata.layers["counts"] = adata.X

regularizedvi.AmbientRegularizedSCVI.setup_anndata(
    adata,
    layer="counts",
    batch_key="batch",
    categorical_covariate_keys=["site", "donor"],
)

### Create model

The key hyperparameters:

- **`n_hidden=512`** — wide hidden layer acts as a dictionary of atomic regulatory programmes that are combined to produce gene expression. High width is needed because these programmes can be granular and covariate-dependent.
- **`n_latent=128`** — large latent space gives the model free capacity, removing competition between cell types for the same latent dimension. For complex datasets with many cell types/states, values up to 1000 are not unreasonable.
- **`n_layers=1`** — single hidden layer. The encoder is simply used for amortised inference.

The following are set as **package defaults** (no need to specify):
- `gene_likelihood='nb'` — Negative Binomial
- `dispersion='gene-batch'` — per-gene, per-batch overdispersion
- `use_additive_background=True` — ambient RNA correction
- `use_batch_in_decoder=False` — batch-free decoder
- `regularise_dispersion=True` — containment prior on dispersion
- `use_batch_norm='none'`, `use_layer_norm='both'` — LayerNorm preferred
- `library_log_vars_weight=0.05` — constrained library prior
- `library_n_hidden=16` — low-capacity library encoder

In [ ]:
model = regularizedvi.AmbientRegularizedSCVI(
    adata,
    n_hidden=512,
    n_layers=1,
    n_latent=128,
)

### Train

- **`train_size=1.0`** — use all data for training. With regularised dispersion and additive background, the model is less prone to overfitting.
- **`max_epochs=2000`** — longer training for convergence. The model has more parameters (additive background, learned library size) that need sufficient epochs.
- **`batch_size=1024`** — large batches for stable gradient estimates, especially important for the per-batch additive background parameters.

In [ ]:
model.train(
    train_size=1.0,
    max_epochs=2000,
    batch_size=1024,
)

### Training loss

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(model.history_["elbo_train"][80:])
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("ELBO")
axes[0].set_title("ELBO (training)")

axes[1].plot(model.history_["reconstruction_loss_train"][80:])
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Reconstruction loss")
axes[1].set_title("Reconstruction loss (training)")

plt.tight_layout()
plt.show()

### Save model

In [ ]:
ref_run_name = f"{results_folder}/regularizedvi_bone_marrow_tutorial"

model.save(ref_run_name, overwrite=True)

## 5. Latent space and visualisation

In [ ]:
latent = model.get_latent_representation()
adata.obsm["X_scVI"] = latent

print(f"Latent representation shape: {latent.shape}")

In [ ]:
# Compute KNN and UMAP
k = 50
sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=k, metric="euclidean")
sc.tl.umap(adata, min_dist=0.4, spread=1.3)
sc.tl.leiden(adata, resolution=12, flavor="igraph")

In [ ]:
# UMAP coloured by batch, site, and cell type annotations
color = ["site", "batch", "l1_cell_type", "l2_cell_type"]

rcParams["figure.figsize"] = 7, 7
sc.pl.umap(
    adata,
    color=color,
    color_map="RdPu",
    ncols=1,
    palette=sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy,
    size=3,
    vmin=0,
    vmax="p99.9",
    gene_symbols="SYMBOL",
    use_raw=False,
    legend_fontsize=10,
)

In [ ]:
# Leiden clusters
rcParams["figure.figsize"] = 7, 7
sc.pl.umap(
    adata,
    color="leiden",
    color_map="RdPu",
    ncols=1,
    palette=sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy,
    size=3,
    legend_fontsize=10,
)

## 6. Marker gene visualisation

Normalise marker genes per cell (counts per 10k) and overlay on UMAP to verify that the latent space captures expected biology.

In [ ]:
markers = {
    "HSC": ["CD34", "SMIM24"],
    "Erythroid": ["HBZ", "HBA1"],
    "T/ILC": ["CD3D"],
    "NK": ["NKG7"],
    "Myeloid": ["CD14", "CSF1"],
    "DC": ["CD1C"],
    "B/Plasma": ["PTPRC"],
    "Cytotoxic": ["GZMB"],
}

for group_name, gene_list in markers.items():
    # Filter to genes present in the data
    present = [g for g in gene_list if g in adata.var["SYMBOL"].values]
    if not present:
        continue

    # Normalise: counts per 10k
    gene_idx = adata.var["SYMBOL"].isin(present)
    selected_expr = adata[:, gene_idx].X.multiply(1.0 / adata.obs[["GEX_n_counts"]].values)
    selected_expr = selected_expr.toarray() * 1e4

    col_names = [f"{m} normalised" for m in present]
    adata.obs[col_names] = selected_expr

    sc.pl.umap(
        adata,
        color=col_names,
        color_map="RdPu",
        ncols=len(col_names),
        size=3,
        vmin=0,
        vmax="p99.99",
        gene_symbols="SYMBOL",
        use_raw=False,
        legend_fontsize=10,
        title=[f"{group_name}: {m}" for m in present],
    )

## 7. Save outputs

In [ ]:
import os

output_dir = f"{ref_run_name}/outputs/"
os.makedirs(output_dir, exist_ok=True)

# Save latent representation
X_scVI = pd.DataFrame(
    adata.obsm["X_scVI"],
    index=adata.obs_names,
    columns=range(adata.obsm["X_scVI"].shape[1]),
)
X_scVI.to_csv(f"{output_dir}/X_scVI.csv")

# Save UMAP coordinates
X_umap = pd.DataFrame(
    adata.obsm["X_umap"],
    index=adata.obs_names,
    columns=range(2),
)
X_umap.to_csv(f"{output_dir}/X_umap_k{k}.csv")

# Save leiden clustering
adata.obs[["leiden"]].to_csv(f"{output_dir}/leiden_k{k}.csv")

# Save KNN graph
scipy.sparse.save_npz(
    f"{output_dir}/distances_euclidean_k{k}.npz",
    adata.obsp["distances"],
    compressed=True,
)
scipy.sparse.save_npz(
    f"{output_dir}/connectivities_euclidean_k{k}.npz",
    adata.obsp["connectivities"],
    compressed=True,
)

print(f"Outputs saved to {output_dir}")

## Summary

This tutorial demonstrated the basic workflow:

1. Load and QC the data
2. Filter genes using `regularizedvi.utils.filter_genes`
3. Set up anndata with `batch_key` and `categorical_covariate_keys`
4. Create and train `AmbientRegularizedSCVI` with recommended hyperparameters
5. Extract latent representation and visualise with UMAP

### When to use regularizedvi

- **Best suited for single-nucleus RNA-seq** (independent modality and multiome), which typically has substantial ambient RNA contamination
- **Study design matters**: the structured assumptions (additive ambient + multiplicative categorical covariates) depend on the experimental design
- **Using as standard scVI with ambient correction**: if you provide the batch covariate to both `batch_key` and `categorical_covariate_keys`, the model effectively operates as standard scVI with ambient RNA correction

See the [distribution comparison notebook](distribution_comparison.ipynb) for a detailed comparison of the NB distribution implementations and prior directions.